In [ ]:
# ============================================================
# DAY 29 — FULL PYTORCH SENTIMENT CLASSIFIER
# ============================================================
#
# TEXT
#   ↓
# TOKENIZATION
#   ↓
# TOKEN IDs
#   ↓
# EMBEDDING
#   ↓
# MEAN POOLING
#   ↓
# LINEAR CLASSIFIER
#   ↓
# POSITIVE / NEGATIVE
#
# ============================================================

import random
import torch
import torch.nn as nn


# ============================================================
# 1. SETTINGS
# ============================================================

SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

EMBEDDING_DIM = 16
LEARNING_RATE = 0.05
EPOCHS = 200


# ============================================================
# 2. OUR DATASET
# ============================================================
#
# label:
#   0 = NEGATIVE
#   1 = POSITIVE
#
# ============================================================

reviews = [

    # ---------------- POSITIVE ----------------

    ("the movie was great", 1),
    ("this movie was amazing", 1),
    ("i loved this film", 1),
    ("the story was wonderful", 1),
    ("the acting was excellent", 1),
    ("what an awesome movie", 1),
    ("this was a fantastic film", 1),
    ("i really enjoyed the movie", 1),
    ("the movie was beautiful", 1),
    ("the characters were great", 1),
    ("the ending was amazing", 1),
    ("this film was enjoyable", 1),
    ("i liked this movie a lot", 1),
    ("the acting was brilliant", 1),
    ("a very good movie", 1),
    ("this was a wonderful experience", 1),
    ("the movie was entertaining", 1),
    ("i would watch this again", 1),
    ("the film was impressive", 1),
    ("absolutely loved this movie", 1),

    # ---------------- NEGATIVE ----------------

    ("the movie was terrible", 0),
    ("this movie was awful", 0),
    ("i hated this film", 0),
    ("the story was boring", 0),
    ("the acting was terrible", 0),
    ("what a horrible movie", 0),
    ("this was a terrible film", 0),
    ("i really disliked the movie", 0),
    ("the movie was disappointing", 0),
    ("the characters were boring", 0),
    ("the ending was awful", 0),
    ("this film was boring", 0),
    ("i did not like this movie", 0),
    ("the acting was horrible", 0),
    ("a very bad movie", 0),
    ("this was a terrible experience", 0),
    ("the movie was extremely boring", 0),
    ("i would never watch this again", 0),
    ("the film was disappointing", 0),
    ("absolutely hated this movie", 0),
]


# ============================================================
# 3. SHUFFLE THE DATA
# ============================================================

random.shuffle(reviews)


# ============================================================
# 4. TRAIN / TEST SPLIT
# ============================================================

split = int(len(reviews) * 0.8)

train_data = reviews[:split]
test_data = reviews[split:]

print("=" * 60)
print("DATASET")
print("=" * 60)

print("Total reviews :", len(reviews))
print("Training      :", len(train_data))
print("Testing       :", len(test_data))


# ============================================================
# 5. BUILD VOCABULARY
# ============================================================
#
# Every word gets an integer ID.
#
# Example:
#
#   movie  -> 12
#   great  -> 7
#   boring -> 4
#
# We also create:
#
#   <UNK> = unknown word
#
# ============================================================

word_counts = {}

for text, label in train_data:

    for word in text.lower().split():

        word_counts[word] = word_counts.get(word, 0) + 1


# Special token for words the model has never seen
stoi = {
    "<UNK>": 0
}


# Add all training words
for word in sorted(word_counts):

    if word not in stoi:

        stoi[word] = len(stoi)


# Reverse dictionary:
# ID -> word

itos = {
    index: word
    for word, index in stoi.items()
}


print("\n" + "=" * 60)
print("VOCABULARY")
print("=" * 60)

print("Vocabulary size:", len(stoi))

print("\nFirst 20 words:")

for word, index in list(stoi.items())[:20]:

    print(f"{word:15} -> {index}")


# ============================================================
# 6. TEXT → TOKEN IDs
# ============================================================

def encode(text):

    words = text.lower().split()

    ids = []

    for word in words:

        if word in stoi:

            ids.append(stoi[word])

        else:

            ids.append(stoi["<UNK>"])

    return torch.tensor(
        ids,
        dtype=torch.long
    )


# ============================================================
# 7. TEST OUR ENCODER
# ============================================================

print("\n" + "=" * 60)
print("TOKENIZATION TEST")
print("=" * 60)

example = "the movie was great"

print("Text:")
print(example)

print("\nToken IDs:")
print(encode(example))


# ============================================================
# 8. THE NEURAL NETWORK
# ============================================================

class SentimentNet(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim
    ):

        super().__init__()

        # ----------------------------------------------------
        # EMBEDDING
        # ----------------------------------------------------
        #
        # Each word ID gets a vector.
        #
        # vocab_size = number of words
        # embedding_dim = size of each vector
        #
        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )


        # ----------------------------------------------------
        # CLASSIFIER
        # ----------------------------------------------------
        #
        # Input:
        #   one sentence vector
        #
        # Output:
        #   2 scores
        #
        #       0 = negative
        #       1 = positive
        #
        self.classifier = nn.Linear(
            embedding_dim,
            2
        )


    # ========================================================
    # FORWARD PASS
    # ========================================================

    def forward(self, ids):

        # ----------------------------------------------------
        # STEP 1:
        # Convert IDs into word vectors
        # ----------------------------------------------------

        vectors = self.embedding(ids)


        # ----------------------------------------------------
        # STEP 2:
        # Average all word vectors
        #
        # This gives us ONE vector representing
        # the entire review.
        # ----------------------------------------------------

        sentence_vector = vectors.mean(
            dim=0,
            keepdim=True
        )


        # ----------------------------------------------------
        # STEP 3:
        # Classify the sentence
        # ----------------------------------------------------

        output = self.classifier(
            sentence_vector
        )


        return output


# ============================================================
# 9. CREATE THE MODEL
# ============================================================

model = SentimentNet(
    vocab_size=len(stoi),
    embedding_dim=EMBEDDING_DIM
)


print("\n" + "=" * 60)
print("MODEL")
print("=" * 60)

print(model)


# ============================================================
# 10. LOSS FUNCTION
# ============================================================

loss_fn = nn.CrossEntropyLoss()


# ============================================================
# 11. OPTIMIZER
# ============================================================

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)


# ============================================================
# 12. FUNCTION TO CALCULATE ACCURACY
# ============================================================

def calculate_accuracy(data):

    correct = 0

    total = len(data)


    # We don't need gradients while testing
    with torch.no_grad():

        for text, label in data:

            ids = encode(text)

            output = model(ids)

            prediction = output.argmax(
                dim=1
            ).item()


            if prediction == label:

                correct += 1


    return correct / total


# ============================================================
# 13. TRAINING
# ============================================================

print("\n" + "=" * 60)
print("TRAINING")
print("=" * 60)


for epoch in range(1, EPOCHS + 1):

    # --------------------------------------------------------
    # Store losses for all training examples
    # --------------------------------------------------------

    losses = []


    for text, label in train_data:

        # ----------------------------------------------
        # Convert text → IDs
        # ----------------------------------------------

        ids = encode(text)


        # ----------------------------------------------
        # Convert label → tensor
        # ----------------------------------------------

        target = torch.tensor(
            [label],
            dtype=torch.long
        )


        # ----------------------------------------------
        # FORWARD
        # ----------------------------------------------

        output = model(ids)


        # ----------------------------------------------
        # LOSS
        # ----------------------------------------------

        loss = loss_fn(
            output,
            target
        )


        losses.append(loss)


        # ----------------------------------------------
        # CLEAR OLD GRADIENTS
        # ----------------------------------------------

        optimizer.zero_grad()


        # ----------------------------------------------
        # BACKPROPAGATION
        # ----------------------------------------------

        loss.backward()


        # ----------------------------------------------
        # UPDATE WEIGHTS
        # ----------------------------------------------

        optimizer.step()


    # --------------------------------------------------------
    # Average loss
    # --------------------------------------------------------

    average_loss = torch.stack(
        losses
    ).mean().item()


    # --------------------------------------------------------
    # Accuracy
    # --------------------------------------------------------

    train_accuracy = calculate_accuracy(
        train_data
    )


    # --------------------------------------------------------
    # Print progress
    # --------------------------------------------------------

    if epoch == 1 or epoch % 10 == 0:

        print(
            f"Epoch {epoch:3d} | "
            f"Loss: {average_loss:.4f} | "
            f"Train accuracy: "
            f"{train_accuracy:.3f}"
        )


# ============================================================
# 14. TEST THE MODEL
# ============================================================

print("\n" + "=" * 60)
print("TEST RESULTS")
print("=" * 60)


test_accuracy = calculate_accuracy(
    test_data
)


print(
    f"Test accuracy: "
    f"{test_accuracy:.3f}"
)


# ============================================================
# 15. SHOW TEST PREDICTIONS
# ============================================================

print("\n" + "=" * 60)
print("TEST PREDICTIONS")
print("=" * 60)


with torch.no_grad():

    for text, actual_label in test_data:

        ids = encode(text)

        output = model(ids)

        prediction = output.argmax(
            dim=1
        ).item()


        actual_name = (
            "POSITIVE"
            if actual_label == 1
            else "NEGATIVE"
        )


        predicted_name = (
            "POSITIVE"
            if prediction == 1
            else "NEGATIVE"
        )


        print(
            f"{predicted_name:8} "
            f"| actual={actual_name:8} "
            f"| {text}"
        )


# ============================================================
# 16. FIND WRONG PREDICTIONS
# ============================================================

print("\n" + "=" * 60)
print("WRONG PREDICTIONS")
print("=" * 60)


wrong_count = 0


with torch.no_grad():

    for text, actual_label in test_data:

        ids = encode(text)

        output = model(ids)

        prediction = output.argmax(
            dim=1
        ).item()


        if prediction != actual_label:

            wrong_count += 1


            actual_name = (
                "POSITIVE"
                if actual_label == 1
                else "NEGATIVE"
            )


            predicted_name = (
                "POSITIVE"
                if prediction == 1
                else "NEGATIVE"
            )


            print(
                "\nText:",
                text
            )

            print(
                "Actual:",
                actual_name
            )

            print(
                "Predicted:",
                predicted_name
            )


print(
    "\nNumber of wrong predictions:",
    wrong_count
)


# ============================================================
# 17. PREDICT A NEW REVIEW
# ============================================================

def predict_sentiment(text):

    model.eval()


    with torch.no_grad():

        ids = encode(text)

        output = model(ids)

        probabilities = torch.softmax(
            output,
            dim=1
        )


        prediction = output.argmax(
            dim=1
        ).item()


    if prediction == 1:

        label = "POSITIVE"

    else:

        label = "NEGATIVE"


    return label, probabilities[0]


# ============================================================
# 18. TRY NEW REVIEWS
# ============================================================

print("\n" + "=" * 60)
print("NEW REVIEW TESTS")
print("=" * 60)


new_reviews = [

    "the movie was amazing",

    "this film was terrible",

    "i loved the acting",

    "the story was boring",

    "what an excellent movie",

    "this was a horrible experience",

    "the movie was fantastic",

    "i hated this film",

]


for text in new_reviews:

    label, probabilities = predict_sentiment(
        text
    )


    print(
        f"\n{text}"
    )

    print(
        "Prediction:",
        label
    )

    print(
        "Negative probability:",
        round(
            probabilities[0].item(),
            3
        )
    )

    print(
        "Positive probability:",
        round(
            probabilities[1].item(),
            3
        )
    )


# ============================================================
# 19. TRY A COMPLETELY NEW SENTENCE
# ============================================================

print("\n" + "=" * 60)
print("UNKNOWN WORD TEST")
print("=" * 60)


text = "the restaurant was fantastic"


label, probabilities = predict_sentiment(
    text
)


print("Review:", text)

print("Prediction:", label)

print(
    "Negative:",
    round(
        probabilities[0].item(),
        3
    )
)

print(
    "Positive:",
    round(
        probabilities[1].item(),
        3
    )
)


# ============================================================
# 20. INSPECT LEARNED EMBEDDINGS
# ============================================================

print("\n" + "=" * 60)
print("LEARNED WORD EMBEDDINGS")
print("=" * 60)


words_to_check = [
    "great",
    "amazing",
    "excellent",
    "terrible",
    "awful",
    "boring",
]


with torch.no_grad():

    for word in words_to_check:

        if word in stoi:

            word_id = stoi[word]

            vector = model.embedding.weight[
                word_id
            ]


            print(
                f"\n{word}:"
            )

            print(
                vector
            )


# ============================================================
# 21. COMPARE TWO WORD VECTORS
# ============================================================

def cosine_similarity(a, b):

    return torch.dot(a, b) / (
        torch.norm(a) *
        torch.norm(b)
    )


print("\n" + "=" * 60)
print("EMBEDDING SIMILARITY")
print("=" * 60)


with torch.no_grad():

    great = model.embedding.weight[
        stoi["great"]
    ]

    amazing = model.embedding.weight[
        stoi["amazing"]
    ]

    terrible = model.embedding.weight[
        stoi["terrible"]
    ]


    sim_positive = cosine_similarity(
        great,
        amazing
    ).item()


    sim_opposite = cosine_similarity(
        great,
        terrible
    ).item()


print(
    "great vs amazing:",
    round(sim_positive, 3)
)


print(
    "great vs terrible:",
    round(sim_opposite, 3)
)


# ============================================================
# 22. SAVE THE MODEL
# ============================================================

torch.save(
    {
        "model_state": model.state_dict(),
        "vocabulary": stoi,
        "embedding_dim": EMBEDDING_DIM
    },
    "sentiment_model.pt"
)


print("\n" + "=" * 60)
print("MODEL SAVED")
print("=" * 60)

print(
    "Saved as: sentiment_model.pt"
)


# ============================================================
# 23. INTERACTIVE MODE
# ============================================================

print("\n" + "=" * 60)
print("INTERACTIVE SENTIMENT CLASSIFIER")
print("=" * 60)

print(
    "Type a review."
)

print(
    "Type 'quit' to stop."
)


while True:

    text = input(
        "\nYour review: "
    )


    if text.lower() == "quit":

        break


    if not text.strip():

        continue


    label, probabilities = predict_sentiment(
        text
    )


    print(
        "AI:",
        label
    )


    print(
        "Negative:",
        round(
            probabilities[0].item(),
            3
        )
    )


    print(
        "Positive:",
        round(
            probabilities[1].item(),
            3
        )
    )


print("\nDone!")

DATASET
Total reviews : 40
Training      : 32
Testing       : 8

VOCABULARY
Vocabulary size: 45

First 20 words:
<UNK>           -> 0
a               -> 1
absolutely      -> 2
acting          -> 3
again           -> 4
amazing         -> 5
awful           -> 6
bad             -> 7
boring          -> 8
brilliant       -> 9
characters      -> 10
did             -> 11
disappointing   -> 12
disliked        -> 13
ending          -> 14
enjoyable       -> 15
entertaining    -> 16
excellent       -> 17
experience      -> 18
extremely       -> 19

TOKENIZATION TEST
Text:
the movie was great

Token IDs:
tensor([36, 30, 39, 21])

MODEL
SentimentNet(
  (embedding): Embedding(45, 16)
  (classifier): Linear(in_features=16, out_features=2, bias=True)
)

TRAINING
Epoch   1 | Loss: 0.7886 | Train accuracy: 0.719
Epoch  10 | Loss: 0.0022 | Train accuracy: 1.000
Epoch  20 | Loss: 0.0006 | Train accuracy: 1.000
Epoch  30 | Loss: 0.0003 | Train accuracy: 1.000
Epoch  40 | Loss: 0.0001 | Train accuracy: 1.00